# Hunyuan3D — тестовая генерация Заргата\n\nНоутбук для запуска в Google Colab. Сначала создаём геометрию в GLB, затем при наличии памяти пробуем добавить текстуру.\n\n**Важно:** выбери GPU в настройках среды выполнения. Бесплатный GPU может быть недоступен; не покупай платный тариф ради этого теста.

In [ ]:
import torch, os, platform\nprint('Python:', platform.python_version())\nprint('PyTorch:', torch.__version__)\nprint('CUDA available:', torch.cuda.is_available())\nif torch.cuda.is_available():\n    print('GPU:', torch.cuda.get_device_name(0))\n    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))\nelse:\n    raise RuntimeError('GPU не выдана. В Colab открой Среда выполнения → Сменить среду выполнения → T4 GPU и запусти ячейку снова.')

## 1. Получаем официальный код\nМы не копируем сторонний репозиторий в OMOBA: скачиваем его непосредственно из официального источника.

In [ ]:
%cd /content\n!git clone --depth 1 https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git\n%cd /content/Hunyuan3D-2\n!pip install -r requirements.txt\n!pip install -e .\n%cd /content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer\n!python setup.py install\n%cd /content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer\n!python setup.py install\n%cd /content/Hunyuan3D-2

## 2. Загрузи исходное изображение\nДля первого прохода используй один чистый фронтальный ракурс персонажа, целиком в кадре, без текста и дополнительных панелей. Лучше PNG с нейтральным или прозрачным фоном.

In [ ]:
from google.colab import files\nfrom PIL import Image\nuploaded = files.upload()\nif not uploaded:\n    raise RuntimeError('Файл не загружен')\ninput_image_path = '/content/' + next(iter(uploaded.keys()))\ndisplay(Image.open(input_image_path).convert('RGB').resize((512, 512)))

## 3. Генерируем геометрию\nИспользуем компактную модель Hunyuan3D-2mini. На выходе будет неанимированный меш.

In [ ]:
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline\nshape_pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(\n    'tencent/Hunyuan3D-2mini',\n    subfolder='hunyuan3d-dit-v2-mini',\n    use_safetensors=True,\n    device='cuda'\n)\nshape_mesh = shape_pipe(image=input_image_path)[0]\nshape_path = '/content/zargat_shape.glb'\nshape_mesh.export(shape_path)\nprint('Готово:', shape_path, 'size MB:', round(os.path.getsize(shape_path)/1024**2, 2))

## 4. Необязательное текстурирование\nЭтот этап требует существенно больше памяти. Если Colab сообщает CUDA out of memory, пропусти его: сначала сохраним форму и оценим её.

In [ ]:
RUN_TEXTURE = False  # поменяй на True только если хватает VRAM\nif RUN_TEXTURE:\n    from hy3dgen.texgen import Hunyuan3DPaintPipeline\n    paint_pipe = Hunyuan3DPaintPipeline.from_pretrained('tencent/Hunyuan3D-2')\n    paint_pipe.enable_model_cpu_offload()\n    textured_mesh = paint_pipe(shape_mesh, image=input_image_path)\n    textured_path = '/content/zargat_textured.glb'\n    textured_mesh.export(textured_path)\n    print('Текстурированная модель:', textured_path)\nelse:\n    print('Текстурирование пропущено. Файл формы:', shape_path)

## 5. Скачиваем GLB\nПередай полученный файл в GitHub-ветку `hunyuan3d-zargat` для следующего этапа: проверка меша, оптимизация, скелет и анимации.

In [ ]:
from google.colab import files\noutput_path = '/content/zargat_textured.glb' if os.path.exists('/content/zargat_textured.glb') else shape_path\nfiles.download(output_path)